In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
import warnings
warnings.filterwarnings("ignore")

# Define the data path
data_path = "data/"

# Load datasets
cities = pd.read_csv(os.path.join(data_path, "Cities.csv"))
conferences = pd.read_csv(os.path.join(data_path, "Conferences.csv"))
mteams = pd.read_csv(os.path.join(data_path, "MTeams.csv"))
wteams = pd.read_csv(os.path.join(data_path, "WTeams.csv"))
mregular_results = pd.read_csv(os.path.join(data_path, "MRegularSeasonCompactResults.csv"))
wregular_results = pd.read_csv(os.path.join(data_path, "WRegularSeasonCompactResults.csv"))
mtourney_results = pd.read_csv(os.path.join(data_path, "MNCAATourneyCompactResults.csv"))
wtourney_results = pd.read_csv(os.path.join(data_path, "WNCAATourneyCompactResults.csv"))
mtourney_seeds = pd.read_csv(os.path.join(data_path, "MNCAATourneySeeds.csv"))
wtourney_seeds = pd.read_csv(os.path.join(data_path, "WNCAATourneySeeds.csv"))

In [2]:
# Extract numeric seed values
def extract_seed(seed):
    return int(seed[1:3]) if isinstance(seed, str) and seed[1:3].isdigit() else np.nan

mtourney_seeds["Seed"] = mtourney_seeds["Seed"].apply(extract_seed)
wtourney_seeds["Seed"] = wtourney_seeds["Seed"].apply(extract_seed)

# Compute advanced team statistics

In [3]:
def compute_team_stats(results):
    team_stats = results.groupby(["Season", "WTeamID"]).agg({
        "WScore": ["mean", "sum"],
        "LScore": ["mean", "sum"],
        "NumOT": "sum"
    }).reset_index()

    # Rename columns
    team_stats.columns = ["Season", "TeamID", "AvgWScore", "TotalWScore", "AvgLScore", "TotalLScore", "OTGames"]
    
    # Compute additional stats
    team_stats["ScoreMargin"] = team_stats["TotalWScore"] - team_stats["TotalLScore"]

    return team_stats

# Compute statistics for men’s and women’s teams
mteam_stats = compute_team_stats(mregular_results)
wteam_stats = compute_team_stats(wregular_results)

# Prepare training data

In [4]:
def prepare_training_data(results, seeds, team_stats):
    results = results.merge(seeds, left_on=["Season", "WTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "WSeed"}).drop(columns=["TeamID"])
    results = results.merge(seeds, left_on=["Season", "LTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "LSeed"}).drop(columns=["TeamID"])

    # Merge team statistics
    results = results.merge(team_stats.rename(columns={"TeamID": "WTeamID"}), on=["Season", "WTeamID"], how="left")
    results = results.merge(team_stats.rename(columns={"TeamID": "LTeamID"}), on=["Season", "LTeamID"], how="left", suffixes=("_W", "_L"))

    # Compute matchup features
    results["SeedDiff"] = results["WSeed"] - results["LSeed"]
    results["WinRateDiff"] = results["AvgWScore_W"] - results["AvgWScore_L"]
    results["ScoreMarginDiff"] = results["ScoreMargin_W"] - results["ScoreMargin_L"]

    # Create win/loss versions of the dataset
    win_features = results[["WTeamID", "LTeamID", "WSeed", "LSeed", "SeedDiff", "WinRateDiff", "ScoreMarginDiff"]].copy()
    win_features["Win"] = 1

    loss_features = results[["LTeamID", "WTeamID", "LSeed", "WSeed", "SeedDiff", "WinRateDiff", "ScoreMarginDiff"]].copy()
    loss_features.columns = ["WTeamID", "LTeamID", "WSeed", "LSeed", "SeedDiff", "WinRateDiff", "ScoreMarginDiff"]
    loss_features["Win"] = 0

    return pd.concat([win_features, loss_features], ignore_index=True)

# Prepare training data
mtrain_data = prepare_training_data(mtourney_results, mtourney_seeds, mteam_stats)
wtrain_data = prepare_training_data(wtourney_results, wtourney_seeds, wteam_stats)
full_train_data = pd.concat([mtrain_data, wtrain_data], ignore_index=True)

In [5]:
# Split & Scale Data
X = full_train_data.drop(columns=["Win"])
y = full_train_data["Win"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

In [6]:


# 📌 Train Models
models = {
    "Logistic Regression": LogisticRegression(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "XGBoost": XGBClassifier()
}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict_proba(X_valid_scaled)[:, 1]
    score = log_loss(y_valid, y_pred)
    print(f"{name} Log Loss: {score}")

# 📌 Create Ensemble Model
ensemble_model = VotingClassifier(
    estimators=[
        ("rf", RandomForestClassifier()),
        ("gb", GradientBoostingClassifier()),
        ("xgb", XGBClassifier())
    ],
    voting="soft"
)

ensemble_model.fit(X_train_scaled, y_train)



Logistic Regression Log Loss: 0.5147594539928143
Random Forest Log Loss: 0.11181745402983437
Gradient Boosting Log Loss: 0.045136865754202
XGBoost Log Loss: 0.04042311310183691


VotingClassifier(estimators=[('rf', RandomForestClassifier()),
                             ('gb', GradientBoostingClassifier()),
                             ('xgb',
                              XGBClassifier(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=None, device=None,
                                            early_stopping_rounds=None,
                                            enable_categorical=False,
                                            eval_metric=None,
                                            feature_types=None, gamma=None,
                                            grow_polic...
                                            importance_type=None,
                                            interaction_constraints=None,
                                            learning_rate=None, max_bin=None,
                                            max_cat_threshold=None,
                                            max_cat_to_onehot=None,
                                            max_delta_step=None, max_depth=None,
                                            max_leaves=None,
                                            min_child_weight=None, missing=nan,
                                            monotone_constraints=None,
                                            multi_strategy=None,
                                            n_estimators=None, n_jobs=None,
                                            num_parallel_tree=None,
                                            random_state=None, ...))],
                 voting='soft')

# 📌 Prepare Submission Data

In [8]:
def prepare_submission_features(submission_df, seeds, team_stats):
    # Extract Season, WTeamID, and LTeamID from ID
    submission_df[["Season", "WTeamID", "LTeamID"]] = submission_df["ID"].str.split("_", expand=True).astype(int)

    # Merge seed information
    submission_df = submission_df.merge(seeds, left_on=["Season", "WTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "WSeed"}).drop(columns=["TeamID"])
    submission_df = submission_df.merge(seeds, left_on=["Season", "LTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "LSeed"}).drop(columns=["TeamID"])

    # 🔍 Check missing values in Seeds
    print("🔍 Missing Seeds Before Filling:")
    print("WSeed NaNs:", submission_df["WSeed"].isna().sum(), " | LSeed NaNs:", submission_df["LSeed"].isna().sum())

    # 🚨 Fill missing seed values with median seed value
    submission_df["WSeed"].fillna(seeds["Seed"].median(), inplace=True)
    submission_df["LSeed"].fillna(seeds["Seed"].median(), inplace=True)

    # Merge team statistics
    submission_df = submission_df.merge(team_stats.rename(columns={"TeamID": "WTeamID"}), on=["Season", "WTeamID"], how="left")
    submission_df = submission_df.merge(team_stats.rename(columns={"TeamID": "LTeamID"}), on=["Season", "LTeamID"], how="left", suffixes=("_W", "_L"))

    # 🔍 Check missing values in Team Stats
    print("🔍 Missing Stats Before Filling:")
    print(submission_df.isna().sum())

    # 🚨 Fill missing stats with median values
    for col in ["AvgWScore_W", "AvgWScore_L", "ScoreMargin_W", "ScoreMargin_L"]:
        submission_df[col].fillna(submission_df[col].median(), inplace=True)

    # Compute matchup features
    submission_df["SeedDiff"] = submission_df["WSeed"] - submission_df["LSeed"]
    submission_df["WinRateDiff"] = submission_df["AvgWScore_W"] - submission_df["AvgWScore_L"]
    submission_df["ScoreMarginDiff"] = submission_df["ScoreMargin_W"] - submission_df["ScoreMargin_L"]

    # Extract final feature set
    submission_features = submission_df[["WTeamID", "LTeamID", "WSeed", "LSeed", "SeedDiff", "WinRateDiff", "ScoreMarginDiff"]]

    # 🔍 Final check for missing values
    print("✅ Missing Values After Fix:", submission_features.isna().sum().sum())

    return submission_df, submission_features


# 📌 Load Submission Data and Apply Fix
submission_df = pd.read_csv(os.path.join(data_path, "SampleSubmissionStage2.csv"))
submission_df, submission_features = prepare_submission_features(submission_df, mtourney_seeds, mteam_stats)

# 📌 Scale Features
submission_features_scaled = scaler.transform(submission_features)

# 📌 Generate Predictions
submission_df["Pred"] = ensemble_model.predict_proba(submission_features_scaled)[:, 1]

# 📌 Save Submission File
submission_df[["ID", "Pred"]].to_csv("submission.csv", index=False)

print("✅ Submission file generated successfully!")

🔍 Missing Seeds Before Filling:
WSeed NaNs: 131407  | LSeed NaNs: 131407
🔍 Missing Stats Before Filling:
ID                   0
Pred                 0
Season               0
WTeamID              0
LTeamID              0
WSeed                0
LSeed                0
AvgWScore_W      65542
TotalWScore_W    65542
AvgLScore_W      65542
TotalLScore_W    65542
OTGames_W        65542
ScoreMargin_W    65542
AvgWScore_L      65503
TotalWScore_L    65503
AvgLScore_L      65503
TotalLScore_L    65503
OTGames_L        65503
ScoreMargin_L    65503
dtype: int64
✅ Missing Values After Fix: 0
✅ Submission file generated successfully!
